In [ ]:
   import os
   os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q -U trl peft bitsandbytes datasets accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
from datasets import load_dataset
dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train")
print(f"Total transactions : {len(dataset):,}")

README.md:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): reconstructing file:   0%|          |  0.00B /  138MB            

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): downloading bytes:           |  0.00B            

...

Cifer-Fraud-Detection-Dataset-AF-part-8-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/21000000 [00:00<?, ? examples/s]

Total transactions : 21,000,000


In [ ]:
dataset = dataset.select(range(min(1_000_000,len(dataset))))
print(f"Working with : {len(dataset):,} transactions")

Working with : 1,000,000 transactions


In [ ]:
fraud_data = dataset.filter(lambda x:x['isFraud'] == 1)
legit_data = dataset.filter(lambda x:x['isFraud'] == 0)

print(f"Fraudulent Transactions : {len(fraud_data):,}")
print(f"Non-Fraudulent Transactions : {len(legit_data):,}")

Filter:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Fraudulent Transactions : 1,313
Non-Fraudulent Transactions : 998,687


In [ ]:
fraud_sample = fraud_data.shuffle(seed=42).select(range(500))
legit_sample = legit_data.shuffle(seed=42).select(range(500))
print(f"Fraudulent Sample : {len(fraud_sample):,}")
print(f"Non-Fraudulent Sample : {len(legit_sample):,}")

Fraudulent Sample : 500
Non-Fraudulent Sample : 500


In [ ]:
from datasets import concatenate_datasets

balanced_dataset = concatenate_datasets([fraud_sample, legit_sample])
balanced_dataset = balanced_dataset.shuffle(seed=42)
print(f"Balanced Dataset : {len(balanced_dataset)} transactions")

Balanced Dataset : 1000 transactions


In [ ]:
def create_conversations(example):
  transaction_type = example['type']
  amount = example['amount']
  old_balance_org = example['oldbalanceOrg']
  new_balance_org = example['newbalanceOrig']
  old_balance_dest = example['oldbalanceDest']
  new_balance_dest = example['newbalanceDest']

  user_msg = f"""Analyze this transaction for fraud risk:
  -Type : {transaction_type}
  -Amount : ${amount:,.2f}
  -Sender Balance Before : ${old_balance_org:,.2f}
  -Sender Balance After : ${new_balance_org:,.2f}
  -Recepient Balance Before : ${old_balance_dest:,.2f}
  -Recepient Balance After : ${new_balance_dest:,.2f}"""

  if example['isFraud'] == 1:
    asst_msg = "HIGH"
  else:
    asst_msg = "LOW"
  return {"messages":[
      {"role" : "user" , "content" : user_msg},
      {"role" : "assistant" , "content" : asst_msg}
  ]}

In [ ]:
train_dataset = balanced_dataset.map(create_conversations)
print(f"Created {len(train_dataset)} training conversations")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Created 1000 training conversations


In [ ]:
print(train_dataset[0]['messages'])

[{'role': 'user', 'content': 'Analyze this transaction for fraud risk:\n  -Type : CASH_IN\n  -Amount : $243.74\n  -Sender Balance Before : $16,101.76\n  -Sender Balance After : $11,824,895.38\n  -Recepient Balance Before : $1,028,631.69\n  -Recepient Balance After : $8.45'}, {'role': 'assistant', 'content': 'LOW'}]


In [ ]:
from transformers import BitsAndBytesConfig
import torch
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
!pip install -q transformers bitsandbytes accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto"
)

print(f"Model loaded: {model_id}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
!pip install -q peft

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
!pip install -q datasets trl

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./fraud-detector",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    fp16=True,
    bf16=False
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args
)

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
   for p in trainer.model.parameters():
       if p.requires_grad:
           p.data = p.data.float()
   import trl.kernels.chunked_logprob as _clp
   _clp._MM_OUT_DTYPE = False

In [ ]:
trainer.train()

Step,Training Loss
10,1.976370
20,0.963931
30,0.803556
40,0.758818
50,0.702904
60,0.696064
70,0.702976
80,0.698561
90,0.692351
100,0.692361


TrainOutput(global_step=375, training_loss=0.7390042661031088, metrics={'train_runtime': 456.3234, 'train_samples_per_second': 6.574, 'train_steps_per_second': 0.822, 'total_flos': 3407825376264192.0, 'train_loss': 0.7390042661031088, 'entropy': 0.690289318561554, 'mean_token_accuracy': 0.7391859650611877, 'num_tokens': 413910.0, 'epoch': 3.0})

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
repo_name = "shloksutar/fraud-detector"
trainer.push_to_hub(repo_name)

CommitInfo(commit_url='https://huggingface.co/shloksutar/fraud-detector/commit/da64e80d47e70acfb30be59365e80a52e5684fbc', commit_message='shloksutar/fraud-detector', commit_description='', oid='da64e80d47e70acfb30be59365e80a52e5684fbc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shloksutar/fraud-detector', endpoint='https://huggingface.co', repo_type='model', repo_id='shloksutar/fraud-detector'), pr_revision=None, pr_num=None)

In [ ]:
repo_name = "shloksutar/fraud-detector"

trainer.push_to_hub(repo_name)

print(f"Uploaded to: https://huggingface.co/{repo_name}")

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded to: https://huggingface.co/shloksutar/fraud-detector


In [ ]:
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 59.5 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline

repo_name = "shloksutar/fraud-detector"

generator = pipeline("text-generation", model=repo_name, device="cuda")

adapter_config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 17.5MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

In [ ]:
question = """Analyze this transaction for fraud risk:
- Type: TRANSFER
- Amount: $85,000.00
- Sender Balance Before: $85,000.00
- Sender Balance After: $0.00
- Recipient Balance Before: $0.00
- Recipient Balance After: $85,000.00"""

output = generator([{"role": "user", "content": question}], max_new_tokens=150, return_full_text=False)[0]
print(output["generated_text"])

LOW
